# Layer 2 Experiment (exploration) — 6 candidate behaviour features

**Part of:** the research project "Scalability of Orchestration in a Security Operation Center" (Kemdiktisaintek)
**Applicable methodology reference:** the project's internal Layer 2 framework document (not included in this repository).
This notebook implements its §2 ("feature computation rules — must be uniform").

**What this notebook DOES (version of 21 Oct 2026):**
1. Compute 6 candidate behaviour features per **monitored host** (section A.2): 4 malware VMs (zeus .250/.251,
   tinba, dridex) + 2 external benign hosts (CTU-Normal-32, CTU-Normal-33). zeus_vm_151 is EXCLUDED.
2. All `conn.log` files come from the same Zeek 8.2.2 (`zeek -C -r <pcap> local`); zeus from the deduplicated pcap.
3. Two IPv6 rule variants applied THE SAME WAY to all hosts: `ipv4_only` and `ipv4_plus_ipv6`
   (link-local IPv6 whose MAC is verified to be the host's). Which variant to use = item #16.
4. Feature stability within **uniform 5-minute** windows for all hosts (section H), and a check of whether
   "inside/outside the malware range" is meaningful (section I).

**What it deliberately does NOT do:** clustering, choosing the final aggregation unit (#7), deciding the IPv6
rule (#16), treating .250/.251 as two independent infections (effective malware N = 3, framework §1b).

> **Status:** this notebook documents Stage 1 of Layer 2, which was later SUPERSEDED by the Behaviour-Class
> clustering in `compute_behavior_classes.py` (see `README.md`).


In [1]:
from pathlib import Path
import sys, re
from typing import Optional
import pandas as pd
import numpy as np

BASE_DIR = Path.cwd()  # run this notebook from inside the layer2-behavior-features/ folder
LAYER1_DIR = BASE_DIR.parent / "layer1-maltrail-zeek"
ZEEK_OUT_DIR = LAYER1_DIR / "data" / "zeek_out"
MALTRAIL_OUT_DIR = LAYER1_DIR / "data" / "maltrail_out"
RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("Looking for Layer 1 data in:", LAYER1_DIR.resolve())
if not LAYER1_DIR.exists():
    print(f"[!] {LAYER1_DIR} not found. This notebook must sit NEXT TO")
    print("    layer1-maltrail-zeek/ (in the same parent folder).")
else:
    print("OK — Experiment 2 (Layer 1) folder found.")

Python: 3.9.6
Looking for Layer 1 data in: /Users/ariefikhwanwicaksono/Disertasi/PFR/beyond-the-blacklist-soc/layer1-maltrail-zeek
OK — Experiment 2 (Layer 1) folder found.


## Section A — pcap manifest

Ground truth here is **per capture**. `family` uses the identity verified at payload level
(framework §1), not the dataset file name. The `zeek_dir` column points to the Zeek 8.2.2 output in
`layer1-maltrail-zeek/data/zeek_out/`.


In [2]:
DATA_DIR = LAYER1_DIR / "data"

PILOT_KEYS = [
    {"key": "zeus", "family": "x0Proto RAT, likely Blackshades or a derivative (dataset label: Zeus)",
     "ground_truth": "malware", "zeek_dir": "zeus_dedup", "pcap": "pcap_derived/zeus_dedup.pcap"},
    {"key": "tinba", "family": "Tinba (or a derivative of the code leaked in 2014)", "ground_truth": "malware",
     "zeek_dir": "tinba", "pcap": "pcap_raw/tinba.pcap"},
    {"key": "dridex", "family": "Dridex", "ground_truth": "malware",
     "zeek_dir": "dridex", "pcap": "pcap_raw/dridex.pcap"},
    {"key": "bittorrent", "family": "BitTorrent (synthetic, IXIA BPS)", "ground_truth": "benign",
     "zeek_dir": "bittorrent", "pcap": "pcap_raw/bittorrent.pcap"},
    {"key": "skype", "family": "Skype (synthetic, IXIA BPS)", "ground_truth": "benign",
     "zeek_dir": "skype", "pcap": "pcap_raw/skype.pcap"},
    {"key": "ctu_normal32", "family": "CTU-Normal-32 (Kali, browsing Quantcast top 401-700)", "ground_truth": "benign",
     "zeek_dir": "ctu_normal32", "pcap": "ctu_normal32/raw/2017-05-02_normal.pcap"},
    {"key": "ctu_normal33", "family": "CTU-Normal-33 (Win7, user web activity)", "ground_truth": "benign",
     "zeek_dir": "ctu_normal33", "pcap": "ctu_normal33/raw/capture-new-Windows-7-Full.pcap"},
]
GROUND_TRUTH = {p["key"]: p["ground_truth"] for p in PILOT_KEYS}
FAMILY = {p["key"]: p["family"] for p in PILOT_KEYS}
PCAP_PATH = {p["key"]: DATA_DIR / p["pcap"] for p in PILOT_KEYS}

# Maltrail was only run (Layer 1) on these 5 pcaps. For other pcaps the alert history is UNKNOWN (NaN),
# NOT 0 -- filling in 0 would turn this feature into a label proxy (benign hosts always 0 by construction).
MALTRAIL_RUN_KEYS = {"zeus", "tinba", "dridex", "bittorrent", "skype"}

for p in PILOT_KEYS:
    print(f"{p['key']:13s} {p['ground_truth']:8s} {p['zeek_dir']:13s} {p['family']}")


zeus          malware  zeus_dedup    x0Proto RAT, likely Blackshades or a derivative (dataset label: Zeus)
tinba         malware  tinba         Tinba (or a derivative of the code leaked in 2014)
dridex        malware  dridex        Dridex
bittorrent    benign   bittorrent    BitTorrent (synthetic, IXIA BPS)
skype         benign   skype         Skype (synthetic, IXIA BPS)
ctu_normal32  benign   ctu_normal32  CTU-Normal-32 (Kali, browsing Quantcast top 401-700)
ctu_normal33  benign   ctu_normal33  CTU-Normal-33 (Win7, user web activity)


## Section A.2 — Population of monitored hosts

Definition: host = **one machine, identified by its MAC** (not every unique `id.orig_h`). IPs used by the
same MAC as the **source of IP packets** are merged. Gateways/routers/NAT and `::` are not hosts.
The IP→MAC mapping is in `HOST_MANIFEST` and is re-checked automatically with `tcpdump -e` (next cell).
Decision history: Layer 2 framework document §1 (17–21 Oct 2026).

| capture | host_id | MAC | IP | note |
|---|---|---|---|---|
| zeus (dedup) | zeus_vm_250, zeus_vm_251 | `08:00:27:3c:8d:c9`, `08:00:27:56:f9:4a` | .250 / .251 | **same sample & C2 → not independent replicates** (same `infection_group`) |
| zeus | ~~zeus_vm_151~~ | `08:00:27:d0:2f:3c` | .151 | **EXCLUDED**: active for 68.8 s, 9 unicast WS-Discovery/WSD flows, 0 alerts |
| tinba | tinba_vm | `08:00:27:fb:bb:39` | 10.0.2.108 + fe80::705a:… | |
| dridex | dridex_vm | `08:00:27:62:20:12` | 10.0.2.104 + fe80::c06e:… | fe80 = 61% of flows (DHCPv6) → see the IPv6 variants |
| ctu_normal32 | ctu_normal32_host | `08:00:27:a3:83:43` | 10.0.2.15 + fe80::69dd:e614:2b2:dfd0 | external benign host, Kali |
| ctu_normal33 | ctu_normal33_host | `08:00:27:40:76:00` | 10.0.2.15 + fe80::353e:6fce:ee32:1a73 | external benign host, Win7 |
| bittorrent, skype | — | — | — | synthetic IXIA BPS, **no host** (per-capture table only) |

**Two IPv6 rule variants, applied THE SAME WAY to all hosts** (framework §2.2; final choice = item #16):
`ipv4_only` = the host's IPv4 address only; `ipv4_plus_ipv6` = plus the link-local IPv6 whose MAC is
verified to be the same. zeus .250/.251 have no IPv6 traffic, so both variants are identical for them.

**zeus uses the deduplicated pcap** (`data/pcap_derived/zeus_dedup.pcap`, script
`layer1-maltrail-zeek/tools/dedup_zeus_pcap.py`): 28,927 frames from MAC `78:e4:00:6c:39:cd` whose IP
content is byte-identical to a VM frame (≤1 s) were dropped. Validation: .250/.251 flow rows stay at
21,891, `resp_pkts` stays at 26,458, `orig_pkts` 57,842 → 28,922, share of even `orig_pkts` 100% → 3.2%.
The zeus Maltrail log still comes from the original pcap (its alerts are not duplicated).

**Disclosure on the zeus features** (framework §1b): 92–98% of the .250/.251 C2 flows are reconnects
rejected by the server (`REJ`), so their `n_conn_rows`, inter-arrival and duration reflect C2 availability.


In [3]:
# One entry per monitored host. "ips" = all addresses used by that MAC as the source of IP packets.
HOST_MANIFEST = {
    "zeus": [  # dedup pcap; zeus_vm_151 EXCLUDED (see the markdown above)
        {"host_id": "zeus_vm_250", "mac": "08:00:27:3c:8d:c9", "ips": ["192.168.0.250"], "infection_group": "x0proto_pair"},
        {"host_id": "zeus_vm_251", "mac": "08:00:27:56:f9:4a", "ips": ["192.168.0.251"], "infection_group": "x0proto_pair"},
    ],
    "tinba": [
        {"host_id": "tinba_vm", "mac": "08:00:27:fb:bb:39",
         "ips": ["10.0.2.108", "fe80::705a:530f:1701:e8a5"], "infection_group": "tinba"},
    ],
    "dridex": [
        {"host_id": "dridex_vm", "mac": "08:00:27:62:20:12",
         "ips": ["10.0.2.104", "fe80::c06e:84b6:bcb8:a750"], "infection_group": "dridex"},
    ],
    "bittorrent": [],  # synthetic IXIA BPS -- no host
    "skype": [],       # synthetic IXIA BPS -- no host
    "ctu_normal32": [
        {"host_id": "ctu_normal32_host", "mac": "08:00:27:a3:83:43",
         "ips": ["10.0.2.15", "fe80::69dd:e614:2b2:dfd0"], "infection_group": "benign_ctu_normal32"},
    ],
    "ctu_normal33": [
        {"host_id": "ctu_normal33_host", "mac": "08:00:27:40:76:00",
         "ips": ["10.0.2.15", "fe80::353e:6fce:ee32:1a73"], "infection_group": "benign_ctu_normal33"},
    ],
}
HOST_POPULATION = {k: ("synthetic_no_host" if not v else ("benign_host" if GROUND_TRUTH[k] == "benign" else "sandbox_vm"))
                   for k, v in HOST_MANIFEST.items()}

# Two IPv6 rule variants -- the SAME function is used for all hosts.
IPV6_RULES = {
    "ipv4_only": lambda ip: ":" not in ip,
    "ipv4_plus_ipv6": lambda ip: True,
}

IP_TO_HOST = {(k, ip): h["host_id"] for k, hosts in HOST_MANIFEST.items() for h in hosts for ip in h["ips"]}
HOST_INFO = {h["host_id"]: h for hosts in HOST_MANIFEST.values() for h in hosts}

for k, hosts in HOST_MANIFEST.items():
    print(f"{k:13s} {HOST_POPULATION[k]:18s} " + ", ".join(f"{h['host_id']} {h['ips']}" for h in hosts))


zeus          sandbox_vm         zeus_vm_250 ['192.168.0.250'], zeus_vm_251 ['192.168.0.251']
tinba         sandbox_vm         tinba_vm ['10.0.2.108', 'fe80::705a:530f:1701:e8a5']
dridex        sandbox_vm         dridex_vm ['10.0.2.104', 'fe80::c06e:84b6:bcb8:a750']
bittorrent    synthetic_no_host  
skype         synthetic_no_host  
ctu_normal32  benign_host        ctu_normal32_host ['10.0.2.15', 'fe80::69dd:e614:2b2:dfd0']
ctu_normal33  benign_host        ctu_normal33_host ['10.0.2.15', 'fe80::353e:6fce:ee32:1a73']


In [4]:
# Re-check HOST_MANIFEST against the pcaps: each host IP must appear as a source ONLY with that host's MAC.
# Needs `tcpdump` (built into macOS). The CTU-Normal-32 pcap is ~860 MB -> this cell can take ~1 minute.
import subprocess, shutil
from collections import defaultdict

VERIFY_MAC_WITH_TCPDUMP = True
# Bridge MAC re-emitting VM frames in zeus; after dedup, 2 TCP frames .250->.151 without a twin remain.
KNOWN_DUPLICATING_MACS = {"zeus": {"78:e4:00:6c:39:cd"}}
SRC_RE = re.compile(r"^\S+ (\S+) > \S+, ethertype IPv[46] .*?: (\S+) > ")

if VERIFY_MAC_WITH_TCPDUMP and shutil.which("tcpdump"):
    for key, hosts in HOST_MANIFEST.items():
        if not hosts:
            continue
        ips = [ip for h in hosts for ip in h["ips"]]
        bpf = " or ".join(f"src host {ip}" for ip in ips)
        out = subprocess.run(["tcpdump", "-nn", "-e", "-r", str(PCAP_PATH[key]), bpf],
                             capture_output=True, text=True).stdout
        macs_by_ip = defaultdict(set)
        for line in out.splitlines():
            m = SRC_RE.match(line)
            if m:
                ip = m.group(2)
                ip = ip.rsplit(".", 1)[0] if (ip.count(".") == 4 or ":" in ip) else ip  # strip the port
                macs_by_ip[ip].add(m.group(1))
        for h in hosts:
            for ip in h["ips"]:
                seen = macs_by_ip.get(ip, set()) - KNOWN_DUPLICATING_MACS.get(key, set())
                status = "OK" if seen == {h["mac"]} else f"!! MISMATCH: {sorted(seen)}"
                print(f"[{key}] {ip:28s} -> {h['mac']}  {status}")
else:
    print("MAC verification skipped (VERIFY_MAC_WITH_TCPDUMP=False or tcpdump not available).")


[zeus] 192.168.0.250                -> 08:00:27:3c:8d:c9  OK
[zeus] 192.168.0.251                -> 08:00:27:56:f9:4a  OK


[tinba] 10.0.2.108                   -> 08:00:27:fb:bb:39  OK


[tinba] fe80::705a:530f:1701:e8a5    -> 08:00:27:fb:bb:39  OK


[dridex] 10.0.2.104                   -> 08:00:27:62:20:12  OK


[dridex] fe80::c06e:84b6:bcb8:a750    -> 08:00:27:62:20:12  OK


[ctu_normal32] 10.0.2.15                    -> 08:00:27:a3:83:43  OK
[ctu_normal32] fe80::69dd:e614:2b2:dfd0     -> 08:00:27:a3:83:43  OK


[ctu_normal33] 10.0.2.15                    -> 08:00:27:40:76:00  OK
[ctu_normal33] fe80::353e:6fce:ee32:1a73    -> 08:00:27:40:76:00  OK


## Section B — Load `conn.log` (Zeek 8.2.2)

The parser is exactly the same as section D.2 of `layer1_experiment.ipynb`. All `conn.log` files here were
produced by Zeek 8.2.2 with the same command (`zeek -C -r <pcap> local`) — including CTU-Normal-32/33, which
were re-processed from the original Stratosphere pcaps rather than using the Bro 2.5.1 logs shipped with the
dataset.


In [5]:
def parse_zeek_log(log_path: Path) -> pd.DataFrame:
    fields, rows = None, []
    with open(log_path) as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith("#fields"):
                fields = line.split("\t")[1:]
            elif line.startswith("#") or not line.strip():
                continue
            else:
                rows.append(line.split("\t"))
    if fields is None:
        raise ValueError(f"No #fields line found in {log_path} — unexpected format.")
    df = pd.DataFrame(rows, columns=fields)
    return df.replace("-", pd.NA)


conn_frames = []
for p in PILOT_KEYS:
    key = p["key"]
    conn_log = ZEEK_OUT_DIR / p["zeek_dir"] / "conn.log"
    if not conn_log.exists():
        print(f"[{key}] no conn.log at {conn_log} -- skipped.")
        continue
    df = parse_zeek_log(conn_log)
    df["pilot_key"] = key
    df["ground_truth"] = GROUND_TRUTH[key]
    conn_frames.append(df)
    print(f"[{key}] {len(df)} flow rows  ({p['zeek_dir']}/conn.log)")

if conn_frames:
    all_conn = pd.concat(conn_frames, ignore_index=True)
    NUM_COLS = ["ts", "duration", "orig_bytes", "resp_bytes", "orig_pkts", "resp_pkts",
                "orig_ip_bytes", "resp_ip_bytes"]
    for c in NUM_COLS:
        if c in all_conn.columns:
            all_conn[c] = pd.to_numeric(all_conn[c], errors="coerce")
    print()
    print("Total combined flow rows:", len(all_conn))
    print("Unique originator IPs (id.orig_h) per capture -- NOT the number of hosts, see section A.2:")
    print(all_conn.groupby("pilot_key")["id.orig_h"].nunique())
else:
    all_conn = pd.DataFrame()
    print("No conn.log could be loaded at all -- check the LAYER1_DIR path above.")


[zeus] 22053 flow rows  (zeus_dedup/conn.log)
[tinba] 10813 flow rows  (tinba/conn.log)
[dridex] 9384 flow rows  (dridex/conn.log)
[bittorrent] 7517 flow rows  (bittorrent/conn.log)
[skype] 6321 flow rows  (skype/conn.log)


[ctu_normal32] 72316 flow rows  (ctu_normal32/conn.log)
[ctu_normal33] 4568 flow rows  (ctu_normal33/conn.log)



Total combined flow rows: 132972
Unique originator IPs (id.orig_h) per capture -- NOT the number of hosts, see section A.2:
pilot_key
bittorrent      7125
ctu_normal32       4
ctu_normal33       4
dridex             4
skype           6025
tinba              4
zeus               4
Name: id.orig_h, dtype: int64


In [6]:
# Audit: which originators are mapped to a host and which are dropped (and how many rows).
if not all_conn.empty:
    all_conn["host_id"] = [IP_TO_HOST.get((k, ip)) for k, ip in zip(all_conn["pilot_key"], all_conn["id.orig_h"])]
    audit = (
        all_conn.assign(status=np.where(all_conn["host_id"].notna(), "host", "dropped"))
        .groupby(["pilot_key", "status"])
        .agg(n_rows=("uid", "size"), n_orig_ip=("id.orig_h", "nunique"))
        .unstack("status", fill_value=0)
    )
    display(audit)
    dropped = all_conn[all_conn["host_id"].isna() & all_conn["pilot_key"].map(HOST_POPULATION).ne("synthetic_no_host")]
    print("Originators dropped in captures with hosts (should be only gateway/NAT/'::'/0.0.0.0, plus zeus .151/.1):")
    display(dropped.groupby(["pilot_key", "id.orig_h"]).size().rename("n_rows").reset_index())
    missing = [(k, ip) for (k, ip) in IP_TO_HOST if not ((all_conn["pilot_key"] == k) & (all_conn["id.orig_h"] == ip)).any()]
    print("Manifest IPs that do NOT appear as originator in conn.log:", missing or "none")


n_rows        n_orig_ip     
status       dropped   host   dropped host
pilot_key                                 
bittorrent      7517      0      7125    0
ctu_normal32       2  72314         2    2
ctu_normal33       2   4566         2    2
dridex             3   9381         2    2
skype           6321      0      6025    0
tinba              3  10810         2    2
zeus             162  21891         2    2

Originators dropped in captures with hosts (should be only gateway/NAT/'::'/0.0.0.0, plus zeus .151/.1):


,pilot_key,id.orig_h,n_rows
0,ctu_normal32,0.0.0.0,1
1,ctu_normal32,::,1
2,ctu_normal33,0.0.0.0,1
3,ctu_normal33,::,1
4,dridex,10.0.2.2,2
5,dridex,::,1
6,tinba,10.0.2.2,2
7,tinba,::,1
8,zeus,192.168.0.1,156
9,zeus,192.168.0.151,6


Manifest IPs that do NOT appear as originator in conn.log: none


## Section C — Load the Maltrail alert history per host

The regex is exactly the same as section C.4 of `layer1_experiment.ipynb`. Used as a **feature**, not a gate.
Only captures in `MALTRAIL_RUN_KEYS` were ever scanned by Maltrail; other hosts get NaN.
The zeus log comes from the ORIGINAL pcap (before dedup); its alert lines are unique, so it is not affected by the duplication.


In [7]:
MALTRAIL_LOG_RE = re.compile(
    r'^"(?P<timestamp>[^"]+)"\s+(?P<hostname>\S+)\s+(?P<src_ip>\S+)\s+(?P<src_port>\d+)\s+'
    r'(?P<dst_ip>\S+)\s+(?P<dst_port>\d+)\s+(?P<proto>\S+)\s+(?P<event_type>\S+)\s+'
    r'(?P<observable>"[^"]*"|\S+)\s+"(?P<label>[^"]+)"\s*\((?P<trail_category>[^)]+)\)\s*$'
)
MALTRAIL_COLUMNS = ["timestamp", "hostname", "src_ip", "src_port", "dst_ip", "dst_port",
                     "proto", "event_type", "observable", "label", "trail_category", "_raw_line"]


def parse_maltrail_log(log_path: Path) -> pd.DataFrame:
    if not log_path.exists():
        return pd.DataFrame(columns=MALTRAIL_COLUMNS)
    rows = []
    with open(log_path, encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line.strip():
                continue
            m = MALTRAIL_LOG_RE.match(line)
            if m:
                d = m.groupdict()
                d["observable"] = d["observable"].strip('"')
                d["_raw_line"] = line
                rows.append(d)
            else:
                rows.append({col: None for col in MALTRAIL_COLUMNS[:-1]} | {"_raw_line": line})
    return pd.DataFrame(rows, columns=MALTRAIL_COLUMNS)


maltrail_frames = []
for p in PILOT_KEYS:
    key = p["key"]
    if key not in MALTRAIL_RUN_KEYS:
        print(f"[{key}] Maltrail was NEVER run on this capture -> alert history = NaN (unknown), not 0")
        continue
    log_path = MALTRAIL_OUT_DIR / f"{key}.log"
    df = parse_maltrail_log(log_path)
    df["pilot_key"] = key
    maltrail_frames.append(df)
    print(f"[{key}] {len(df)} Maltrail alert lines" + ("" if log_path.exists() else " (no file -> 0 alerts, by design)"))

all_maltrail = pd.concat(maltrail_frames, ignore_index=True) if maltrail_frames else pd.DataFrame(columns=MALTRAIL_COLUMNS)

if not all_maltrail.empty and all_maltrail["src_ip"].notna().any():
    alerts = all_maltrail.dropna(subset=["src_ip"]).copy()
    alerts["host_id"] = [IP_TO_HOST.get((k, ip)) for k, ip in zip(alerts["pilot_key"], alerts["src_ip"])]
    unmapped = alerts[alerts["host_id"].isna()]
    if not unmapped.empty:
        print("Alerts whose src_ip is outside the monitored hosts (not counted as a host feature):")
        display(unmapped.groupby(["pilot_key", "src_ip"]).size().rename("n_alert").reset_index())
    maltrail_per_host = (
        alerts.dropna(subset=["host_id"])
        .groupby(["pilot_key", "host_id"])
        .size()
        .rename("maltrail_alert_count")
        .reset_index()
    )
else:
    maltrail_per_host = pd.DataFrame(columns=["pilot_key", "host_id", "maltrail_alert_count"])

print()
print("Alert history per host:")
display(maltrail_per_host) if not maltrail_per_host.empty else print("(empty — no alerts with src_ip in this pilot corpus yet)")

[zeus] 5697 Maltrail alert lines
[tinba] 10902 Maltrail alert lines
[dridex] 0 Maltrail alert lines (no file -> 0 alerts, by design)
[bittorrent] 0 Maltrail alert lines (no file -> 0 alerts, by design)
[skype] 0 Maltrail alert lines (no file -> 0 alerts, by design)
[ctu_normal32] Maltrail was NEVER run on this capture -> alert history = NaN (unknown), not 0
[ctu_normal33] Maltrail was NEVER run on this capture -> alert history = NaN (unknown), not 0

Alert history per host:


,pilot_key,host_id,maltrail_alert_count
0,tinba,tinba_vm,10902
1,zeus,zeus_vm_250,2698
2,zeus,zeus_vm_251,2999


## Section D — Compute the 6 candidate features, per (capture, host, IPv6 variant)

Basic aggregation unit here: **one monitored host (`host_id`, section A.2) within one capture, for one IPv6
rule variant**, using the flows that host **originates**. Extra column: `active_hours` (time span of that
host's flows) — `n_conn_rows` and `n_unique_dst` are NOT normalised by duration, so compare them with care
(or use section H, which is per window). Not per flow, not per time window, and not per raw `id.orig_h`. This
is a sensible choice because the SOC's detection unit is ultimately the host, but it remains **one option**,
not a final decision (see section G). Section E provides a per-pcap aggregation (coarser) as a comparison.

Operational definition of each feature — written out explicitly so it can be criticised/replaced, not hidden
in the code:

1. **`bytes_ratio_log2`** — `log2((total orig_bytes + 1) / (total resp_bytes + 1))` per host.
   Positive = the host sends more bytes than it receives.
2. **`dst_port_entropy`** — Shannon entropy (base 2) of the distribution of `id.resp_p` contacted by that
   host. High = ports spread out (many different ports); low = concentrated on a few ports.
3. **`interarrival_cv`** — coefficient of variation (std/mean) of the gaps between consecutive connections
   (`ts`, sorted) of that host. Needs &ge; 3 connections; with fewer it is set to `NaN` (not 0 — do not
   silently assume "perfect regularity" when there is simply too little data). Low CV = more regular gaps (a
   candidate C2 *beaconing* indicator; the direction follows the literature but no threshold is set yet —
   **this needs further empirical verification**).
4. **`n_unique_dst`** — number of distinct `(id.resp_h, id.resp_p)` pairs contacted by that host (not just the
   number of flow rows, which is recorded separately as `n_conn_rows`).
5. **`avg_duration_sec`** — mean `duration` (seconds) of that host's flows, ignoring empty values.
6. **`direction_ratio`** — proportion of flows in which `orig_pkts > resp_pkts` (the host sends more packets
   than its peer). A literal operationalisation, not yet matched to the exact definition of Qu et al. 2015 —
   **this needs verification** before being used as the final definition in the paper.

Diagnostic column (NOT a candidate feature): **`frac_multicast_linklocal`** — proportion of that host's flows
whose destination is multicast, link-local or broadcast (heuristic: private IPv4 ending in `.255` is treated
as a /24 directed broadcast). It shows when a host's features are dominated by local OS noise. Example: 5,696
of the 9,381 flows of the dridex host are DHCPv6 solicits to `ff02::1:2`.

Plus the Maltrail history features (`maltrail_alert_count`, `maltrail_ever_alerted`) and
`ground_truth`/`family` (per pcap, carried as is — see the caveat in section G).

In [8]:
def shannon_entropy(counts: pd.Series) -> float:
    probs = counts / counts.sum()
    return float(-(probs * np.log2(probs)).sum())


def interarrival_cv(ts_sorted: np.ndarray) -> float:
    if len(ts_sorted) < 3:
        return np.nan
    deltas = np.diff(ts_sorted)
    deltas = deltas[deltas > 0]  # drop duplicates of identical timestamps (delta 0)
    if len(deltas) < 2 or deltas.mean() == 0:
        return np.nan
    return float(deltas.std(ddof=1) / deltas.mean())


import ipaddress


def is_local_scope_dst(ip: str) -> bool:
    """Multicast, link-local or broadcast -- traffic that does not leave the local segment."""
    try:
        a = ipaddress.ip_address(ip)
    except ValueError:
        return False
    if a.is_multicast or a.is_link_local:
        return True
    return a.version == 4 and (str(a) == "255.255.255.255" or (a.is_private and str(a).endswith(".255")))


def compute_host_features(group: pd.DataFrame) -> pd.Series:
    orig_bytes = group["orig_bytes"].fillna(0)
    resp_bytes = group["resp_bytes"].fillna(0)
    orig_pkts = group["orig_pkts"].fillna(0)
    resp_pkts = group["resp_pkts"].fillna(0)

    bytes_ratio_log2 = np.log2((orig_bytes.sum() + 1) / (resp_bytes.sum() + 1))

    port_counts = group["id.resp_p"].dropna().value_counts()
    dst_port_entropy = shannon_entropy(port_counts) if len(port_counts) > 0 else np.nan

    ts_sorted = np.sort(group["ts"].dropna().to_numpy())
    cv = interarrival_cv(ts_sorted)

    n_unique_dst = group.dropna(subset=["id.resp_h", "id.resp_p"]).drop_duplicates(
        subset=["id.resp_h", "id.resp_p"]
    ).shape[0]

    avg_duration_sec = group["duration"].dropna().mean() if group["duration"].notna().any() else np.nan

    directional = (orig_pkts > resp_pkts)
    direction_ratio = float(directional.mean()) if len(directional) > 0 else np.nan

    return pd.Series({
        "n_conn_rows": len(group),
        "bytes_ratio_log2": bytes_ratio_log2,
        "dst_port_entropy": dst_port_entropy,
        "interarrival_cv": cv,
        "n_unique_dst": n_unique_dst,
        "avg_duration_sec": avg_duration_sec,
        "direction_ratio": direction_ratio,
        "frac_multicast_linklocal": float(group["id.resp_h"].map(is_local_scope_dst).mean()),
    })


if all_conn.empty:
    host_features = pd.DataFrame()
    print("all_conn is empty -> this cell is skipped.")
else:
    frames = []
    for rule, allowed in IPV6_RULES.items():
        sub = all_conn[all_conn["host_id"].notna() & all_conn["id.orig_h"].map(allowed)]
        hf = sub.groupby(["pilot_key", "host_id"]).apply(compute_host_features).reset_index()
        span = sub.groupby(["pilot_key", "host_id"])["ts"].agg(lambda s: (s.max() - s.min()) / 3600).rename("active_hours")
        hf = hf.merge(span.reset_index(), on=["pilot_key", "host_id"])
        hf["ipv6_rule"] = rule
        hf["host_ips"] = hf["host_id"].map(lambda h: ",".join(ip for ip in HOST_INFO[h]["ips"] if allowed(ip)))
        frames.append(hf)
    host_features = pd.concat(frames, ignore_index=True)
    host_features["family"] = host_features["pilot_key"].map(FAMILY)
    host_features["ground_truth"] = host_features["pilot_key"].map(GROUND_TRUTH)
    host_features["infection_group"] = host_features["host_id"].map(lambda h: HOST_INFO[h]["infection_group"])

    host_features = host_features.merge(maltrail_per_host, on=["pilot_key", "host_id"], how="left")
    scanned = host_features["pilot_key"].isin(MALTRAIL_RUN_KEYS)
    host_features.loc[scanned, "maltrail_alert_count"] = host_features.loc[scanned, "maltrail_alert_count"].fillna(0)
    host_features["maltrail_alert_count"] = host_features["maltrail_alert_count"].astype("Int64")  # NaN = not scanned
    host_features["maltrail_ever_alerted"] = (host_features["maltrail_alert_count"] > 0).astype("boolean")

    cols_order = ["ipv6_rule", "pilot_key", "family", "ground_truth", "infection_group", "host_id", "host_ips",
                  "active_hours", "n_conn_rows", "bytes_ratio_log2", "dst_port_entropy", "interarrival_cv",
                  "n_unique_dst", "avg_duration_sec", "direction_ratio", "frac_multicast_linklocal",
                  "maltrail_alert_count", "maltrail_ever_alerted"]
    host_features = host_features[cols_order].sort_values(["ipv6_rule", "ground_truth", "pilot_key", "host_id"],
                                                          ascending=[False, False, True, True]).reset_index(drop=True)

    print(f"{len(host_features)} rows (IPv6 variant, capture, host):")
    display(host_features)

    out_path = RESULTS_DIR / "host_features_pilot.csv"
    host_features.to_csv(out_path, index=False)
    print("Saved to:", out_path)


/var/folders/tw/24sm5rq11vg62k6ngnn50z1c0000gn/T/ipykernel_17717/3759458227.py:72: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  hf = sub.groupby(["pilot_key", "host_id"]).apply(compute_host_features).reset_index()


12 rows (IPv6 variant, capture, host):


/var/folders/tw/24sm5rq11vg62k6ngnn50z1c0000gn/T/ipykernel_17717/3759458227.py:72: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  hf = sub.groupby(["pilot_key", "host_id"]).apply(compute_host_features).reset_index()


,ipv6_rule,pilot_key,family,ground_truth,infection_group,host_id,host_ips,active_hours,n_conn_rows,bytes_ratio_log2,dst_port_entropy,interarrival_cv,n_unique_dst,avg_duration_sec,direction_ratio,frac_multicast_linklocal,maltrail_alert_count,maltrail_ever_alerted
0,ipv4_plus_ipv6,dridex,Dridex,malware,dridex,dridex_vm,"10.0.2.104,fe80::c06e:84b6:bcb8:a750",675.544517,9381.0,-0.253195,1.252175,0.725121,28.0,42.595923,0.732971,0.607718,0,False
1,ipv4_plus_ipv6,tinba,Tinba (or a derivative of the code leaked in 2...,malware,tinba,tinba_vm,"10.0.2.108,fe80::705a:530f:1701:e8a5",0.485687,10810.0,-1.508789,0.022872,4.633673,7.0,0.071685,0.135893,0.001573,10902,True
2,ipv4_plus_ipv6,zeus,"x0Proto RAT, likely Blackshades or a derivativ...",malware,x0proto_pair,zeus_vm_250,192.168.0.250,9.053936,10919.0,-2.300104,1.535151,1.704353,26.0,1.264506,0.078945,0.007510,2698,True
3,ipv4_plus_ipv6,zeus,"x0Proto RAT, likely Blackshades or a derivativ...",malware,x0proto_pair,zeus_vm_251,192.168.0.251,9.051259,10972.0,-3.512565,1.619344,2.157567,37.0,1.358211,0.053591,0.008020,2999,True
4,ipv4_plus_ipv6,ctu_normal32,"CTU-Normal-32 (Kali, browsing Quantcast top 40...",benign,benign_ctu_normal32,ctu_normal32_host,"10.0.2.15,fe80::69dd:e614:2b2:dfd0",3.951440,72314.0,-4.373596,1.247608,5.096826,3566.0,23.672375,0.141231,0.000774,<NA>,<NA>
5,ipv4_plus_ipv6,ctu_normal33,"CTU-Normal-33 (Win7, user web activity)",benign,benign_ctu_normal33,ctu_normal33_host,"10.0.2.15,fe80::353e:6fce:ee32:1a73",1.146956,4566.0,-4.252373,1.734751,3.933140,527.0,19.928646,0.323040,0.034823,<NA>,<NA>
6,ipv4_only,dridex,Dridex,malware,dridex,dridex_vm,10.0.2.104,286.656348,3682.0,-1.790038,0.717323,1.466368,24.0,5.575682,0.319663,0.000543,0,False
7,ipv4_only,tinba,Tinba (or a derivative of the code leaked in 2...,malware,tinba,tinba_vm,10.0.2.108,0.484270,10803.0,-1.520916,0.014344,4.637179,4.0,0.043326,0.135333,0.000926,10902,True
8,ipv4_only,zeus,"x0Proto RAT, likely Blackshades or a derivativ...",malware,x0proto_pair,zeus_vm_250,192.168.0.250,9.053936,10919.0,-2.300104,1.535151,1.704353,26.0,1.264506,0.078945,0.007510,2698,True
9,ipv4_only,zeus,"x0Proto RAT, likely Blackshades or a derivativ...",malware,x0proto_pair,zeus_vm_251,192.168.0.251,9.051259,10972.0,-3.512565,1.619344,2.157567,37.0,1.358211,0.053591,0.008020,2999,True


Saved to: /Users/ariefikhwanwicaksono/Disertasi/PFR/beyond-the-blacklist-soc/layer2-behavior-features/results/host_features_pilot.csv


## Section E — Summary per host & per capture

The per-host table below contains 4 malware VMs + 2 benign hosts for both IPv6 variants. The per-capture
table (all flows of the capture, without a host filter) still contains bittorrent/skype with the flag
`synthetic_no_host`; their temporal features are not meaningful as host behaviour.
**Not a hypothesis test** — effective malware N = 3, benign = 2.


In [9]:
FEATURE_COLS = ["bytes_ratio_log2", "dst_port_entropy", "interarrival_cv",
                "n_unique_dst", "avg_duration_sec", "direction_ratio", "maltrail_alert_count"]

if host_features.empty:
    print("host_features is empty -> this cell is skipped.")
else:
    for rule in IPV6_RULES:
        print(f"--- PER HOST, variant {rule} ---")
        display(host_features[host_features["ipv6_rule"] == rule]
                .set_index(["ground_truth", "host_id"])[["active_hours", "n_conn_rows"] + FEATURE_COLS].round(3))

    print()
    print("--- PER CAPTURE (all flows, no host filter -- coarse comparison) ---")
    pcap_features = all_conn.groupby("pilot_key").apply(compute_host_features).reset_index()
    pcap_features["ground_truth"] = pcap_features["pilot_key"].map(GROUND_TRUTH)
    pcap_features["host_population"] = pcap_features["pilot_key"].map(HOST_POPULATION)
    pcap_features["zeek_dir"] = pcap_features["pilot_key"].map({p["key"]: p["zeek_dir"] for p in PILOT_KEYS})
    display(pcap_features.set_index("pilot_key").round(3))

    out_path2 = RESULTS_DIR / "pcap_features_pilot.csv"
    pcap_features.to_csv(out_path2, index=False)
    print("Saved to:", out_path2)


--- PER HOST, variant ipv4_only ---


active_hours  n_conn_rows  bytes_ratio_log2  \
ground_truth host_id                                                          
malware      dridex_vm               286.656       3682.0            -1.790   
             tinba_vm                  0.484      10803.0            -1.521   
             zeus_vm_250               9.054      10919.0            -2.300   
             zeus_vm_251               9.051      10972.0            -3.513   
benign       ctu_normal32_host         3.949      72279.0            -4.374   
             ctu_normal33_host         1.147       4515.0            -4.284   

                                dst_port_entropy  interarrival_cv  \
ground_truth host_id                                                
malware      dridex_vm                     0.717            1.466   
             tinba_vm                      0.014            4.637   
             zeus_vm_250                   1.535            1.704   
             zeus_vm_251                   1.619            2.158   
benign       ctu_normal32_host             1.242            5.173   
             ctu_normal33_host             1.664            4.035   

                                n_unique_dst  avg_duration_sec  \
ground_truth host_id                                             
malware      dridex_vm                  24.0             5.576   
             tinba_vm                    4.0             0.043   
             zeus_vm_250                26.0             1.265   
             zeus_vm_251                37.0             1.358   
benign       ctu_normal32_host        3562.0            23.655   
             ctu_normal33_host         520.0            19.025   

                                direction_ratio  maltrail_alert_count  
ground_truth host_id                                                   
malware      dridex_vm                    0.320                     0  
             tinba_vm                     0.135                 10902  
             zeus_vm_250                  0.079                  2698  
             zeus_vm_251                  0.054                  2999  
benign       ctu_normal32_host            0.141                  <NA>  
             ctu_normal33_host            0.315                  <NA>

--- PER HOST, variant ipv4_plus_ipv6 ---


active_hours  n_conn_rows  bytes_ratio_log2  \
ground_truth host_id                                                          
malware      dridex_vm               675.545       9381.0            -0.253   
             tinba_vm                  0.486      10810.0            -1.509   
             zeus_vm_250               9.054      10919.0            -2.300   
             zeus_vm_251               9.051      10972.0            -3.513   
benign       ctu_normal32_host         3.951      72314.0            -4.374   
             ctu_normal33_host         1.147       4566.0            -4.252   

                                dst_port_entropy  interarrival_cv  \
ground_truth host_id                                                
malware      dridex_vm                     1.252            0.725   
             tinba_vm                      0.023            4.634   
             zeus_vm_250                   1.535            1.704   
             zeus_vm_251                   1.619            2.158   
benign       ctu_normal32_host             1.248            5.097   
             ctu_normal33_host             1.735            3.933   

                                n_unique_dst  avg_duration_sec  \
ground_truth host_id                                             
malware      dridex_vm                  28.0            42.596   
             tinba_vm                    7.0             0.072   
             zeus_vm_250                26.0             1.265   
             zeus_vm_251                37.0             1.358   
benign       ctu_normal32_host        3566.0            23.672   
             ctu_normal33_host         527.0            19.929   

                                direction_ratio  maltrail_alert_count  
ground_truth host_id                                                   
malware      dridex_vm                    0.733                     0  
             tinba_vm                     0.136                 10902  
             zeus_vm_250                  0.079                  2698  
             zeus_vm_251                  0.054                  2999  
benign       ctu_normal32_host            0.141                  <NA>  
             ctu_normal33_host            0.323                  <NA>


--- PER CAPTURE (all flows, no host filter -- coarse comparison) ---


/var/folders/tw/24sm5rq11vg62k6ngnn50z1c0000gn/T/ipykernel_17717/1360760348.py:14: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pcap_features = all_conn.groupby("pilot_key").apply(compute_host_features).reset_index()


,n_conn_rows,bytes_ratio_log2,dst_port_entropy,interarrival_cv,n_unique_dst,avg_duration_sec,direction_ratio,frac_multicast_linklocal,ground_truth,host_population,zeek_dir
pilot_key,,,,,,,,,,,
bittorrent,7517.0,0.586,-0.000,0.859,7113.0,0.000,0.002,0.000,benign,synthetic_no_host,bittorrent
ctu_normal32,72316.0,-4.374,1.248,5.097,3567.0,23.672,0.141,0.001,benign,benign_host,ctu_normal32
ctu_normal33,4568.0,-4.252,1.739,3.934,528.0,19.924,0.323,0.035,benign,benign_host,ctu_normal33
dridex,9384.0,-0.253,1.256,0.725,30.0,42.586,0.733,0.608,malware,sandbox_vm,dridex
skype,6321.0,-4.850,-0.000,4.158,6022.0,0.147,0.037,0.000,benign,synthetic_no_host,skype
tinba,10813.0,-1.257,0.027,4.634,9.0,0.249,0.136,0.002,malware,sandbox_vm,tinba
zeus,22053.0,-2.166,1.647,2.062,52.0,1.850,0.073,0.008,malware,sandbox_vm,zeus_dedup


Saved to: /Users/ariefikhwanwicaksono/Disertasi/PFR/beyond-the-blacklist-soc/layer2-behavior-features/results/pcap_features_pilot.csv


## Section F — Feasibility check of `interarrival_cv` per host

Feature #3 needs &ge; 3 connections per host to be meaningful. This cell shows how many hosts have enough
data and how many do not — if most are `NaN`, it signals that the per-host aggregation window (especially for
pcaps with few connections per host) may be too narrow for this feature in particular, even if it is enough
for the others.

In [10]:
if host_features.empty:
    print("host_features is empty -> this cell is skipped.")
else:
    n_total = len(host_features)
    n_valid_cv = host_features["interarrival_cv"].notna().sum()
    print(f"interarrival_cv filled for {n_valid_cv}/{n_total} rows (variant, capture, host).")
    display(host_features.loc[host_features["interarrival_cv"].isna(), ["ipv6_rule", "pilot_key", "host_id", "n_conn_rows"]])


interarrival_cv filled for 12/12 rows (variant, capture, host).


,ipv6_rule,pilot_key,host_id,n_conn_rows


## Section H — Feature stability within 5-minute windows (the SAME length for all hosts)

**Why 5 minutes**: the shortest host is tinba (active ±29 minutes). 5-minute windows give tinba 5 full
windows — the minimum that still makes sense for computing a spread — while the other hosts get more windows
of the SAME length (CTU-Normal-33 ±13, CTU-Normal-32 ±47, zeus ±108, dridex ±8,100). 10-minute windows would
give tinba only 2 full windows (the spread cannot be computed sensibly); 1–2-minute windows too often contain
<10 flows for sparse hosts (dridex). Same window length → spread metrics are comparable across hosts
(framework §2.6).

Rules: windows are counted from the host's first flow; only **full windows**; a window is **valid** if it
contains ≥ `MIN_FLOWS_PER_WINDOW` (=10) flows. Spread metrics per host:
- **SD** for log/bounded variables whose mean can be close to zero: `bytes_ratio_log2`,
  `dst_port_entropy`, `direction_ratio` (CV is not well defined here — framework §2.6).
- **CV** (SD/mean) only for positive variables: `n_conn_rows`, `n_unique_dst`, `avg_duration_sec`,
  `interarrival_cv`.
- Plus the median and P10–P90 per host (used in section I).

Per-window features are computed with the same `compute_host_features` (section D). Per window, `n_conn_rows`
and `n_unique_dst` are already normalised by duration (all windows are 5 minutes).


In [11]:
WINDOW_SEC = 300
MIN_FLOWS_PER_WINDOW = 10
FEATS_SD = ["bytes_ratio_log2", "dst_port_entropy", "direction_ratio"]
FEATS_CV = ["n_conn_rows", "n_unique_dst", "avg_duration_sec", "interarrival_cv"]
WIN_FEATS = FEATS_CV + FEATS_SD

win_rows, win_meta = [], []
for rule, allowed in IPV6_RULES.items():
    sub = all_conn[all_conn["host_id"].notna() & all_conn["id.orig_h"].map(allowed)]
    for (key, hid), g in sub.groupby(["pilot_key", "host_id"]):
        t0 = g["ts"].min()
        n_full = int((g["ts"].max() - t0) // WINDOW_SEC)  # window number n_full is not full -> dropped
        widx = ((g["ts"] - t0) // WINDOW_SEC).astype(int)
        n_valid = 0
        for w, gw in g.groupby(widx):
            if w >= n_full:
                continue
            f = compute_host_features(gw)
            valid = len(gw) >= MIN_FLOWS_PER_WINDOW
            n_valid += valid
            win_rows.append({"ipv6_rule": rule, "pilot_key": key, "host_id": hid, "ground_truth": GROUND_TRUTH[key],
                             "window": w, "valid": valid, **f.to_dict()})
        win_meta.append({"ipv6_rule": rule, "host_id": hid, "n_windows_full": n_full, "n_windows_valid": n_valid})

window_features = pd.DataFrame(win_rows)
window_features.to_csv(RESULTS_DIR / "window_features_5min.csv", index=False)

stab = []
for (rule, hid), g in window_features[window_features["valid"]].groupby(["ipv6_rule", "host_id"]):
    row = {"ipv6_rule": rule, "host_id": hid, "ground_truth": g["ground_truth"].iat[0]}
    for f in WIN_FEATS:
        v = g[f].dropna()
        row[f"{f}__median"] = v.median()
        row[f"{f}__p10"] = v.quantile(0.10)
        row[f"{f}__p90"] = v.quantile(0.90)
        if f in FEATS_SD:
            row[f"{f}__SD"] = v.std(ddof=1)
        else:
            row[f"{f}__CV"] = v.std(ddof=1) / v.mean() if v.mean() > 0 else np.nan
    stab.append(row)
window_stability = pd.DataFrame(stab).merge(pd.DataFrame(win_meta), on=["ipv6_rule", "host_id"], how="right")
window_stability.to_csv(RESULTS_DIR / "window_stability_5min.csv", index=False)

show = ["ipv6_rule", "ground_truth", "host_id", "n_windows_full", "n_windows_valid"] + \
       [f"{f}__CV" for f in FEATS_CV] + [f"{f}__SD" for f in FEATS_SD]
display(window_stability[show].sort_values(["ipv6_rule", "ground_truth", "host_id"], ascending=[False, False, True]).round(3))
print("Saved: results/window_features_5min.csv, results/window_stability_5min.csv")


,ipv6_rule,ground_truth,host_id,n_windows_full,n_windows_valid,n_conn_rows__CV,n_unique_dst__CV,avg_duration_sec__CV,interarrival_cv__CV,bytes_ratio_log2__SD,dst_port_entropy__SD,direction_ratio__SD
8,ipv4_plus_ipv6,malware,dridex_vm,8106,70,0.223,0.218,0.547,0.429,4.807,0.311,0.199
9,ipv4_plus_ipv6,malware,tinba_vm,5,5,0.286,0.194,0.456,0.691,0.021,0.013,0.045
10,ipv4_plus_ipv6,malware,zeus_vm_250,108,108,0.239,0.392,1.237,0.141,1.125,0.336,0.133
11,ipv4_plus_ipv6,malware,zeus_vm_251,108,107,0.255,0.547,2.620,0.256,1.412,0.430,0.087
6,ipv4_plus_ipv6,benign,ctu_normal32_host,47,47,0.282,0.304,0.459,0.365,0.496,0.121,0.085
7,ipv4_plus_ipv6,benign,ctu_normal33_host,13,13,1.007,0.934,0.877,0.432,2.848,0.259,0.101
2,ipv4_only,malware,dridex_vm,3439,70,0.227,0.245,0.715,0.503,4.604,0.291,0.233
3,ipv4_only,malware,tinba_vm,5,5,0.277,0.161,0.166,0.702,0.010,0.006,0.042
4,ipv4_only,malware,zeus_vm_250,108,108,0.239,0.392,1.237,0.141,1.125,0.336,0.133
5,ipv4_only,malware,zeus_vm_251,108,107,0.255,0.547,2.620,0.256,1.412,0.430,0.087


Saved: results/window_features_5min.csv, results/window_stability_5min.csv


## Section I — Is "inside/outside the malware range" meaningful?

An earlier mistake: concluding "inside the range = similar to malware" without checking whether the range
itself is narrow/meaningful. Here, for each feature (5-minute windows, valid ones only), three checks:

1. **Host-level range**: [min, max] of the window medians of the 4 malware VMs (and of the 3 independent
   infection groups: .250/.251 averaged into one), and whether the benign hosts' window medians fall inside it.
2. **Range vs internal variation**: `range_to_within` = width of the between-host malware range ÷ the median
   width of P10–P90 *within* the malware hosts. Both extremes make "inside the range" weak:
   < 1 → the range is narrower than the noise within one host; ≫ 1 → the malware hosts are very heterogeneous
   and the range is so wide that "inside" does not mean "similar". The decisive measure is therefore check 3
   (window-level overlap), not check 1.
3. **Window-level overlap**: proportion of a benign host's windows falling inside the union of the malware
   hosts' P10–P90 intervals (`benign_in_malware_env`), and conversely the proportion of malware windows (mean
   per host, equal weight) inside the union of the benign hosts' P10–P90 intervals (`malware_in_benign_env`).
   Both low → the feature separates; both high → it does not.

With N = 3 malware infections and 2 benign hosts, this is **descriptive**, not a statistical test.


In [12]:
def range_check(rule):
    wf = window_features[(window_features["ipv6_rule"] == rule) & window_features["valid"]]
    ws = window_stability[window_stability["ipv6_rule"] == rule].set_index("host_id")
    mal_hosts = [h for h in ws.index if ws.loc[h, "ground_truth"] == "malware"]
    ben_hosts = [h for h in ws.index if ws.loc[h, "ground_truth"] == "benign"]
    groups = {}
    for h in mal_hosts:
        groups.setdefault(HOST_INFO[h]["infection_group"], []).append(h)
    out = []
    for f in WIN_FEATS:
        med = ws[f"{f}__median"]
        mal_med = med[mal_hosts]
        grp_med = pd.Series({g: med[hs].mean() for g, hs in groups.items()})
        lo, hi = mal_med.min(), mal_med.max()
        within = (ws.loc[mal_hosts, f"{f}__p90"] - ws.loc[mal_hosts, f"{f}__p10"]).median()
        env_m = [(ws.loc[h, f"{f}__p10"], ws.loc[h, f"{f}__p90"]) for h in mal_hosts]
        env_b = [(ws.loc[h, f"{f}__p10"], ws.loc[h, f"{f}__p90"]) for h in ben_hosts]
        inside = lambda v, env: any(a <= v <= b for a, b in env)
        row = {"ipv6_rule": rule, "feature": f,
               "malware_host_range": f"[{lo:.3g}, {hi:.3g}]",
               "malware_group_range": f"[{grp_med.min():.3g}, {grp_med.max():.3g}]",
               "range_to_within": (hi - lo) / within if within > 0 else np.nan}
        for h in ben_hosts:
            row[f"{h}__median"] = med[h]
            row[f"{h}__in_host_range"] = lo <= med[h] <= hi
            vals = wf.loc[wf["host_id"] == h, f].dropna()
            row[f"{h}__in_malware_env"] = vals.map(lambda v: inside(v, env_m)).mean()
        row["malware_in_benign_env"] = np.mean([
            wf.loc[wf["host_id"] == h, f].dropna().map(lambda v: inside(v, env_b)).mean() for h in mal_hosts])
        out.append(row)
    return pd.DataFrame(out)

range_table = pd.concat([range_check(r) for r in IPV6_RULES], ignore_index=True)
range_table.to_csv(RESULTS_DIR / "range_check_5min.csv", index=False)
for rule in IPV6_RULES:
    print(f"--- variant {rule} ---")
    display(range_table[range_table["ipv6_rule"] == rule].drop(columns="ipv6_rule").set_index("feature").round(3))
print("Saved: results/range_check_5min.csv")


--- variant ipv4_only ---


,malware_host_range,malware_group_range,range_to_within,ctu_normal32_host__median,ctu_normal32_host__in_host_range,ctu_normal32_host__in_malware_env,ctu_normal33_host__median,ctu_normal33_host__in_host_range,ctu_normal33_host__in_malware_env,malware_in_benign_env
feature,,,,,,,,,,
n_conn_rows,"[19, 1.87e+03]","[19, 1.87e+03]",36.251,1542.000,True,0.702,138.000,True,0.231,0.654
n_unique_dst,"[3, 9]","[3, 9]",2.353,160.000,False,0.000,36.000,False,0.000,0.018
avg_duration_sec,"[0.0424, 17.5]","[0.0424, 17.5]",8.131,22.507,False,0.915,22.792,False,0.692,0.161
interarrival_cv,"[0.843, 2.32]","[0.843, 2.32]",2.320,3.438,False,0.894,2.714,False,0.923,0.693
bytes_ratio_log2,"[-1.52, 0.767]","[-1.52, 0.652]",1.379,-4.384,False,0.000,-2.910,False,0.462,0.608
dst_port_entropy,"[0.0111, 1.5]","[0.0111, 1.34]",2.899,1.204,True,1.000,1.412,True,1.000,0.179
direction_ratio,"[0.0252, 0.911]","[0.0281, 0.911]",4.439,0.103,True,0.957,0.319,True,0.538,0.339


--- variant ipv4_plus_ipv6 ---


,malware_host_range,malware_group_range,range_to_within,ctu_normal32_host__median,ctu_normal32_host__in_host_range,ctu_normal32_host__in_malware_env,ctu_normal33_host__median,ctu_normal33_host__in_host_range,ctu_normal33_host__in_malware_env,malware_in_benign_env
feature,,,,,,,,,,
n_conn_rows,"[20, 1.88e+03]","[20, 1.88e+03]",36.506,1546.000,True,0.766,139.000,True,0.231,0.654
n_unique_dst,"[3, 10]","[4, 10]",2.333,155.000,False,0.000,37.000,False,0.000,0.007
avg_duration_sec,"[0.0792, 18.4]","[0.0792, 18.4]",8.568,23.033,False,0.979,24.684,False,0.615,0.186
interarrival_cv,"[0.91, 2.32]","[0.91, 2.32]",1.738,3.568,False,0.872,2.654,False,1.000,0.711
bytes_ratio_log2,"[-1.51, 0.837]","[-1.51, 0.837]",1.419,-4.393,False,0.000,-2.897,False,0.462,0.593
dst_port_entropy,"[0.0189, 1.5]","[0.0189, 1.34]",2.630,1.217,True,1.000,1.581,False,0.846,0.422
direction_ratio,"[0.0252, 0.913]","[0.0281, 0.913]",4.449,0.104,True,0.957,0.324,True,0.308,0.335


Saved: results/range_check_5min.csv


## Section G — Summary & what is still open

Decision reference & current status: the project's internal Layer 2 framework document (§4–§6). In short,
still open:

1. **Final aggregation unit (#7)** — per host per capture vs per window. Section H shows how stable the
   features are in 5-minute windows; per capture is not normalised by duration.
2. **IPv6 rule (#16)** — both variants are computed in parallel in all tables.
3. **Small N & pseudo-replication** — malware 4 VMs / 3 independent infections (`infection_group`), benign
   2 hosts from the same Stratosphere lab. All numbers are descriptive.
4. **Benign vs malware confound** — the benign hosts have a user/browser; the malware sandboxes do not.
   Features that separate the classes may reflect that, not the "maliciousness" of the traffic.
5. **zeus features** — deduplicated, but still dominated by `REJ` reconnects to the C2 (section A.2).
6. **`maltrail_alert_count`** — NaN for CTU-Normal (not scanned by Maltrail at the time). Do not fill in 0.
7. **`direction_ratio`** — definition not yet matched to Qu et al. 2015.